In [ ]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.cluster import SpectralClustering
from sklearn.decomposition import PCA

import plotly.express as px


# ------------------------------------------------------------
# Config
# ------------------------------------------------------------

INPUT_FILE = "fighter_career_stats.csv"

K_CLUSTERS = 5
MIN_FIGHTS = 3
N_NEIGHBORS = 15
RANDOM_STATE = 100


# ------------------------------------------------------------
# 1. Load data
# ------------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

df_model = df[df["total_fights"] >= MIN_FIGHTS].copy()

print("Original fighters:", len(df))
print("Fighters after min-fight filter:", len(df_model))


# ------------------------------------------------------------
# 2. Select clustering features
# ------------------------------------------------------------

exclude_cols = [
    "fighter",            # ID column
    "total_fights",       # sample-size / reliability column
    "total_fight_time"    # career-volume column
]

feature_cols = [
    col for col in df_model.columns
    if col not in exclude_cols
]

feature_cols = [
    col for col in feature_cols
    if pd.api.types.is_numeric_dtype(df_model[col])
]

print("Number of clustering features:", len(feature_cols))


# ------------------------------------------------------------
# 3. Preprocess features
# ------------------------------------------------------------

X_raw = df_model[feature_cols].replace([np.inf, -np.inf], np.nan)

preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])

X_scaled = preprocess.fit_transform(X_raw)


# ------------------------------------------------------------
# 4. Spectral clustering
# ------------------------------------------------------------

spectral = SpectralClustering(
    n_clusters=K_CLUSTERS,
    affinity="nearest_neighbors",
    n_neighbors=N_NEIGHBORS,
    assign_labels="kmeans",
    random_state=RANDOM_STATE
)

df_model["style_cluster"] = spectral.fit_predict(X_scaled)


# ------------------------------------------------------------
# 5. Cluster counts and summary
# ------------------------------------------------------------

cluster_counts = df_model["style_cluster"].value_counts().sort_index()

cluster_summary = (
    df_model
    .groupby("style_cluster")[feature_cols]
    .mean()
    .round(3)
)

cluster_summary_table = cluster_summary.T
cluster_summary_table.index.name = "feature"

cluster_summary_table.columns = [
    f"Cluster {col}" for col in cluster_summary_table.columns
]

print("\nCluster counts:")
print(cluster_counts)

display(cluster_summary_table)






Original fighters: 2682
Fighters after min-fight filter: 1855
Number of clustering features: 53

Cluster counts:
style_cluster
0    257
1    782
2    386
3    133
4    117
5    180
Name: count, dtype: int64


,Cluster 0,Cluster 1,Cluster 2,Cluster 3,Cluster 4,Cluster 5
feature,,,,,,
fight_quickness,0.658,0.706,0.681,0.529,0.505,0.729
win_DEC_pct,0.398,0.500,0.453,0.387,0.109,0.494
win_KO_TKO_pct,0.305,0.323,0.163,0.400,0.317,0.292
win_SUB_pct,0.171,0.134,0.333,0.211,0.237,0.098
loss_DEC_pct,0.474,0.505,0.495,0.235,0.315,0.535
loss_KO_TKO_pct,0.346,0.288,0.271,0.437,0.492,0.325
loss_SUB_pct,0.174,0.185,0.217,0.242,0.189,0.117
TOTAL_STR_acc,0.582,0.504,0.586,0.645,0.543,0.475
SIG_STR_acc,0.482,0.442,0.448,0.532,0.423,0.440


In [39]:
# ------------------------------------------------------------
# 5. Add readable cluster style labels
# ------------------------------------------------------------

cluster_style_names = {
    0: "Clinch Pressure Fighter",
    1: "Defensive Distance Striker",
    2: "Submission Grappler",
    3: "Dominant Control Wrestler",
    4: "High-Risk Early-Finish Grappler",
    5: "High-Volume Distance Striker"
}

df_model["style_label"] = df_model["style_cluster"].map(cluster_style_names)
# ------------------------------------------------------------
# 6. 3D PCA visualization with Plotly
# ------------------------------------------------------------

pca = PCA(n_components=3, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

plot_df = pd.DataFrame({
    "fighter": df_model["fighter"].values,
    "style_cluster": df_model["style_cluster"].values,
    "style_label": df_model["style_label"].values,
    "total_fights": df_model["total_fights"].values,
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "PC3": X_pca[:, 2]
})

fig = px.scatter_3d(
    plot_df,
    x="PC1",
    y="PC2",
    z="PC3",
    color="style_label",
    hover_name="fighter",
    hover_data={
        "style_cluster": True,
        "total_fights": True,
        "PC1": ":.3f",
        "PC2": ":.3f",
        "PC3": ":.3f"
    },
    title=f"3D PCA Visualization of Spectral Fighter Style Clusters, k={K_CLUSTERS}"
)

fig = px.scatter_3d(
    plot_df,
    x="PC1",
    y="PC2",
    z="PC3",
    color="style_label",
    hover_name="fighter",
    hover_data={
        "style_cluster": True,
        "total_fights": True,
        "PC1": ":.3f",
        "PC2": ":.3f",
        "PC3": ":.3f"
    },
    title=f"3D PCA Visualization of Spectral Fighter Style Clusters, k={K_CLUSTERS}",
    width=1000,
    height=800
)

fig.update_traces(
    marker=dict(
        size=3,
        opacity=1.0
    )
)

fig.show(
    config={
        "scrollZoom": True,
        "displayModeBar": True
    }
)

In [8]:
def get_fighter_style(fighter_name, df_model):
    """
    Search for a fighter and return their assigned cluster/style.
    Allows partial, case-insensitive search.
    """

    matches = df_model[
        df_model["fighter"]
        .str.lower()
        .str.contains(fighter_name.lower(), na=False)
    ]

    if matches.empty:
        print(f"No fighter found matching: {fighter_name}")
        return None

    result = matches[
        [
            "fighter",
            "style_cluster",
            "style_label",
            "total_fights"
        ]
    ].sort_values("fighter")

    return result

In [ ]:
get_fighter_style("Daniel Cormier", df_model)

prin

,fighter,style_cluster,style_label,total_fights
582,Daniel Cormier,0,Clinch Pressure Fighter,15


In [57]:
# ------------------------------------------------------------
# 6. Save fighter style lookup
# ------------------------------------------------------------

fighter_styles = df_model[["fighter", "style_cluster"]].copy()

fighter_styles["style_label"] = fighter_styles["style_cluster"].map(cluster_style_names)

fighter_styles = fighter_styles.sort_values("fighter").reset_index(drop=True)

print("\nFighter style lookup:")
display(fighter_styles.head())



Fighter style lookup:


,fighter,style_cluster,style_label
0,AJ Dobson,1,Defensive Distance Striker
1,AJ Fletcher,1,Defensive Distance Striker
2,Aaron Phillips,4,High-Risk Early-Finish Grappler
3,Aaron Riley,0,Clinch Pressure Fighter
4,Aaron Rosa,0,Clinch Pressure Fighter


Saved: fighter_style_clusters.csv
